# Sentiment Signal
The Trends signal measures consumer interest - how many people are searching for "quiet luxury" or "zara haul". That's useful, but it's only one side of the picture. News headlines are a different kind of signal: they reflect what analysts, journalists, and investors are actually saying about these companies right now.


I'm pulling Google News headlines for each company, scores them with VADER, and blends the result with the Trends signal to get a composite z-score. The composite then tilts the same max-Sharpe weights from before.

The main catch: Google News RSS only goes back around like 90 days. So unlike the Trends signal which covers 2020 - 2025, the sentiment layer is genuinely recent - for most of the backtest period it simply falls back to Trends-only. That's a limitation worth keeping in mind.

In [ ]:
import time
import warnings
import feedparser
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.optimize import minimize
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

TICKER_LABELS = {
    'MC.PA'   : 'LVMH',
    'ITX.MC'  : 'Inditex',
    'HM-B.ST' : 'H&M',
    'TPR'     : 'Tapestry',
}

# search queries for google trends 
NEWS_QUERIES = {
    'LVMH'     : 'LVMH stock',
    'Inditex'  : 'Inditex Zara stock',
    'H&M'      : 'H&M stock',
    'Tapestry' : 'Tapestry Coach stock',
}

RF_RATE  = 0.04
ALPHA    = 0.3
WINDOW   = 4    # rss 

prices = pd.read_csv('data/prices.csv', index_col=0, parse_dates=True)
print('Prices:', prices.shape)

trends = pd.read_csv('data/trends.csv', index_col=0, parse_dates=True)
prices.columns = [TICKER_LABELS.get(c, c) for c in prices.columns]

print('Trends:', trends.shape)

Prices: (1550, 4)
Trends: (72, 3)


## 1. Headlines

No account, no API key feedparser can handle the parsing. The query format is just a URL:


https://news.google.com/rss/search?q=LVMH+stock&hl=en-US&gl=US&ceid=US:en


I'm being a bit specific with the queries ("LVMH stock" rather than just "LVMH") to reduce irrelevant lifestyle articles and focus on coverage that's actually likely to move the price.

queries: dict of {company_name: search_query}

In [2]:
NEWS_RSS = 'https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en'

def fetch_news(queries, delay=1.5):
    
    rows = []
    for company, query in queries.items():
        url  = NEWS_RSS.format(query=query.replace(' ', '+'))
        feed = feedparser.parse(url)
        entries = feed.get('entries', [])

        if not entries:
            print(f'[warn] {company}: 0 entries returned.')
            continue

        for entry in entries:
            rows.append({
                'company'  : company,
                'published': entry.get('published', ''),
                'title'    : entry.get('title', ''),
            })

        print(f'{company}: {len(entries)} headlines')
        time.sleep(delay)

    if not rows:
        return pd.DataFrame(columns=['company', 'published', 'title'])

    df = pd.DataFrame(rows)
    df['published'] = pd.to_datetime(df['published'], utc=True, errors='coerce')
    df['published'] = df['published'].dt.tz_localize(None)   # drop tz for easy resampling
    df = df.dropna(subset=['published'])
    return df.sort_values('published').reset_index(drop=True)


headlines = fetch_news(NEWS_QUERIES)
print(f'\nTotal headlines: {len(headlines)}')
headlines.head(8)

LVMH: 100 headlines
Inditex: 97 headlines
H&M: 102 headlines
Tapestry: 86 headlines

Total headlines: 385


,company,published,title
0,Inditex,2012-08-17 07:00:00,Fashion chain Zara helps Inditex lift first qu...
1,Inditex,2015-08-05 07:00:00,Inditex Surge Takes Zara Owner’s Market Value ...
2,Tapestry,2017-05-22 07:00:00,Coach Is Buying Kate Spade in Latest Bid for G...
3,Tapestry,2017-10-11 07:00:00,Coach Inc. to Rebrand as Tapestry Inc. - hypeb...
4,Tapestry,2017-10-11 07:00:00,"Coach (COH) to Change Name to Tapestry, Inc., ..."
5,Tapestry,2017-10-11 07:00:00,"Coach changes name to Tapestry, shares sink - ..."
6,Tapestry,2017-10-11 07:00:00,Coach’s Rebrand as Tapestry Mocked on Stock Ma...
7,Inditex,2018-01-04 08:00:00,Amancio Ortega reasserts control over Zara emp...
